# Ferroptosis Hub Genes & Apoptosis in Ovarian Cancer
## Exploratory Data Analysis & Visualisation Notebook

**Author:** Shruti Banerjee  
**Institution:** Department of Bioinformatics, Pondicherry University  
**Degree:** MSc Bioinformatics (2022–2024)  
**Supervisor:** Dr. Basant K. Tiwary

---

This notebook reproduces the key data analysis and visualisations from the MSc dissertation:  
*"Identification of ferroptosis-related hub genes and their potential relation with apoptosis in Ovarian Cancer"*

### What this notebook covers
1. Dataset overview and QC summary
2. HISAT2 alignment score analysis
3. Differential gene expression filtering
4. PPI network statistics
5. Hub gene interaction analysis
6. KEGG pathway enrichment visualisation
7. Co-expression network analysis
8. Key findings summary

### Requirements
```
pip install matplotlib numpy pandas seaborn
```

---
## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Colour palette — consistent throughout notebook
PURPLE = '#534AB7'
TEAL   = '#1D9E75'
CORAL  = '#D85A30'
AMBER  = '#BA7517'
GRAY   = '#888780'
LIGHT  = '#D3D1C7'

plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.25,
    'grid.linestyle'   : '--',
    'figure.dpi'       : 120
})

print('Libraries loaded successfully.')

---
## 1. Dataset Overview

RNA-Seq data was retrieved from the European Nucleotide Archive (ENA SRA).  
Two bioprojects were used — one for diseased ovarian cancer tissue and one for healthy controls.

| Bioproject | Type | Platform | Library |
|---|---|---|---|
| PRJNA1005317 | Diseased (BRCA1/2 mutation) | Illumina HiSeq 300 | Paired-end RNA-Seq |
| PRJNA578440 | Healthy control | Illumina HiSeq 300 | Paired-end RNA-Seq |

In [ ]:
# ── Load sample data ──────────────────────────────────────────────────────────
# If sample_ids.csv is in the data/ folder:
try:
    samples = pd.read_csv('../data/sample_ids.csv')
except FileNotFoundError:
    # Build from thesis data if CSV not found
    samples = pd.DataFrame({
        'sra_run_id': [
            'SRR25637180','SRR25637181','SRR25637182','SRR25637185','SRR25637188',
            'SRR25637205','SRR25637208','SRR25637216','SRR25637217','SRR25637220',
            'SRR25637221','SRR25637222','SRR25637223','SRR25637224','SRR25637225',
            'SRR10313349','SRR10313350','SRR10313351','SRR10313352','SRR10313353',
            'SRR10313354','SRR10313355','SRR10313356','SRR10313357','SRR10313358'
        ],
        'sample_type': ['Diseased']*15 + ['Control']*10,
        'qc_passed': [
            False,True,True,False,False,False,True,True,True,True,
            False,False,False,True,False,
            True,True,True,True,True,True,False,False,False,False
        ],
        'hisat2_alignment_score': [
            None,97.23,97.56,None,None,None,42.36,96.88,97.48,97.86,
            None,None,None,56.23,None,
            96.30,96.29,96.48,96.26,96.42,96.54,None,None,None,None
        ],
        'included_in_analysis': [
            False,True,True,False,False,False,False,True,True,True,
            False,False,False,False,False,
            True,True,True,True,True,True,False,False,False,False
        ]
    })

print(f'Total samples in dataset: {len(samples)}')
print(f'Diseased samples: {len(samples[samples.sample_type=="Diseased"])}')
print(f'Control samples:  {len(samples[samples.sample_type=="Control"])}')
print(f'\nSamples passing all filters:')
print(samples[samples.included_in_analysis==True][['sra_run_id','sample_type','hisat2_alignment_score']])

---
## 2. QC Pipeline — Sample Attrition

Samples were filtered through two sequential steps:
1. **FastQC quality filter** — removed samples with poor base quality, high duplication, or overrepresented sequences
2. **HISAT2 alignment threshold** — removed samples with alignment score below 80%

In [ ]:
# QC attrition table
qc_summary = pd.DataFrame({
    'Stage'   : ['Raw data', 'After FastQC filter', 'After alignment filter'],
    'Diseased': [15, 7, 5],
    'Control' : [12, 6, 6]
})

print('Sample count through QC pipeline:')
print(qc_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(qc_summary))
w = 0.35
b1 = ax.bar(x - w/2, qc_summary['Diseased'], w, color=CORAL, label='Diseased', zorder=3)
b2 = ax.bar(x + w/2, qc_summary['Control'],  w, color=TEAL,  label='Control',  zorder=3)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
            str(int(b.get_height())), ha='center', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(qc_summary['Stage'])
ax.set_ylabel('Number of samples')
ax.set_ylim(0, 19)
ax.set_title('Sample attrition through QC and alignment pipeline', fontsize=12, pad=10)
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. HISAT2 Alignment Score Analysis

HISAT2 (v2.2.1) was used to align paired-end reads to the human reference genome (ENSEMBL GRCh38.13).  
Samples with alignment score < 80% were excluded — two diseased samples (SRR25637208 at 42.36% and SRR25637224 at 56.23%) failed this threshold.

In [ ]:
# Alignment scores for samples that passed FastQC
align_data = pd.DataFrame({
    'sample'  : ['SRR181','SRR182','SRR208*','SRR216','SRR217','SRR220','SRR224*',
                 'SRR349','SRR350','SRR351','SRR352','SRR353','SRR354'],
    'score'   : [97.23,97.56,42.36,96.88,97.48,97.86,56.23,
                 96.30,96.29,96.48,96.26,96.42,96.54],
    'type'    : ['Diseased']*7 + ['Control']*6,
    'passed'  : [True,True,False,True,True,True,False,
                 True,True,True,True,True,True]
})

print(f"Mean alignment score (Diseased, passed): "
      f"{align_data[(align_data.type=='Diseased')&(align_data.passed)]['score'].mean():.2f}%")
print(f"Mean alignment score (Control, passed):  "
      f"{align_data[(align_data.type=='Control')&(align_data.passed)]['score'].mean():.2f}%")
print(f"Samples excluded (score < 80%): {len(align_data[~align_data.passed])}")

colors = [CORAL if (t=='Diseased' and p) else
          '#F09595' if (t=='Diseased' and not p) else
          TEAL
          for t, p in zip(align_data.type, align_data.passed)]

fig, ax = plt.subplots(figsize=(12, 4.5))
bars = ax.bar(align_data.sample, align_data.score, color=colors, zorder=3)
ax.axhline(80, color='red', linestyle='--', alpha=0.7, label='80% threshold')
for bar, val in zip(bars, align_data.score):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5,
            f'{val}%', ha='center', fontsize=7.5, rotation=45)
ax.set_ylabel('Alignment score (%)')
ax.set_ylim(0, 108)
ax.set_title('HISAT2 alignment scores (samples marked * were excluded)', fontsize=12, pad=10)
legend_items = [
    mpatches.Patch(color=CORAL,   label='Diseased (passed)'),
    mpatches.Patch(color='#F09595', label='Diseased (excluded)'),
    mpatches.Patch(color=TEAL,    label='Control (passed)'),
    mpatches.Patch(color='red',   label='80% threshold', linestyle='--')
]
ax.legend(handles=legend_items, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Differential Gene Expression Analysis

DESeq2 (Bioconductor, R) was used to identify differentially expressed genes between healthy and diseased samples.  
The analysis used the reference GTF from ENSEMBL GRCh38.13 to generate gene counts via StringTie.

**Total DEGs identified with reference GTF: 7,862**

In [ ]:
# DEG filtering summary
deg_data = pd.DataFrame({
    'filter'  : ['Total DEGs\n(ref GTF)', 'Significant\n(p<0.05)', 
                 'Highly significant\n(p<0.01)', 'Ferroptosis +\nApoptosis genes'],
    'count'   : [7862, 8002, 6577, 289],
    'color'   : [LIGHT, TEAL, TEAL, CORAL]
})

print('DEG filtering summary:')
for _, row in deg_data.iterrows():
    print(f"  {row['filter'].replace(chr(10),' '):45s} {row['count']:>6,}")

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(deg_data.filter, deg_data['count'], color=deg_data.color, width=0.5, zorder=3)
for bar, val in zip(bars, deg_data['count']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f'{val:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of genes')
ax.set_title('Differential gene expression — filtering funnel', fontsize=12, pad=10)
plt.tight_layout()
plt.show()

---
## 5. PPI Network Statistics

Protein-Protein Interaction (PPI) networks were constructed using the STRING database (v11.5) and visualised in Cytoscape with the NetworkAnalyzer plugin.

Three networks were constructed:
- All differentially expressed genes
- Ferroptosis-specific DEGs
- Co-expressed ferroptosis + apoptosis hub genes

In [ ]:
# PPI network statistics
ppi_data = pd.DataFrame({
    'network'     : ['All DEGs', 'Ferroptosis DEGs', 'Co-expressed hub genes'],
    'nodes'       : [236, 174, 12],
    'interactions': [886, 1469, 25]
})

ppi_data['density'] = (ppi_data.interactions / 
                       (ppi_data.nodes * (ppi_data.nodes - 1) / 2) * 100).round(2)

print('PPI Network Statistics:')
print(ppi_data.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].bar(ppi_data.network, ppi_data.nodes, color=[TEAL, CORAL, PURPLE], zorder=3)
for i, v in enumerate(ppi_data.nodes):
    axes[0].text(i, v+3, str(v), ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Nodes per network', fontsize=11)
axes[0].set_ylabel('Number of nodes')
axes[0].tick_params(axis='x', labelsize=9)

axes[1].bar(ppi_data.network, ppi_data.interactions, color=[TEAL, CORAL, PURPLE], zorder=3)
for i, v in enumerate(ppi_data.interactions):
    axes[1].text(i, v+10, str(v), ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Interactions per network', fontsize=11)
axes[1].set_ylabel('Number of interactions')
axes[1].tick_params(axis='x', labelsize=9)

fig.suptitle('PPI network statistics across three constructed networks', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Hub Gene Analysis

From 289 ferroptosis + apoptosis DEGs, hub genes were identified by their degree (number of interactions) in the PPI network. The top 10 ferroptosis hub genes were ranked by interaction score.

In [ ]:
# Ferroptosis hub genes from PPI network
hub_genes = pd.DataFrame({
    'gene'    : ['TP53','HIF1A','EGFR','IL6','STAT3','PARP1','GPX4','SIRT1','MTOR','NFE2L2'],
    'rank'    : [1,2,3,4,4,6,7,8,8,8],
    'score'   : [10,9,8,7,7,6,5,4,4,4],
    'category': ['Tumour suppressor','Hypoxia','Growth signalling','Inflammation',
                 'Inflammation','DNA repair','Ferroptosis regulator',
                 'Metabolism','Metabolism','Oxidative stress']
})

print('Top 10 Ferroptosis Hub Genes:')
print(hub_genes[['rank','gene','category']].to_string(index=False))

# Co-expressed hub genes from network comparison
coexp_hub = pd.DataFrame({
    'rank' : [1,2,3,4,5,5,5,5,9,10],
    'gene' : ['CDKN1A','JUN','CCL2','GDF15','BTG2','FOSL1','IER3','CYCS','HSPA1B','S100A11'],
    'type' : ['Ferroptosis','Apoptosis','Apoptosis','Ferroptosis','Apoptosis',
              'Apoptosis','Apoptosis','Apoptosis','Apoptosis','Apoptosis'],
    'interactions': [23,18,15,10,8,8,8,8,5,4]
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: ferroptosis PPI hub genes
cat_colors = {
    'Tumour suppressor':CORAL,'Hypoxia':AMBER,'Growth signalling':PURPLE,
    'Inflammation':'#D4537E','DNA repair':GRAY,
    'Ferroptosis regulator':TEAL,'Metabolism':'#639922','Oxidative stress':'#185FA5'
}
colors_h = [cat_colors[c] for c in hub_genes.category]
hub_sorted = hub_genes.sort_values('score')
axes[0].barh(hub_sorted.gene, hub_sorted.score,
             color=[cat_colors[c] for c in hub_sorted.category], height=0.6, zorder=3)
axes[0].set_xlabel('Interaction score')
axes[0].set_title('Ferroptosis hub gene rankings\n(PPI network)', fontsize=11)
axes[0].grid(axis='x', alpha=0.25)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Right: co-expressed hub gene degrees
colors_c = [CORAL if t=='Ferroptosis' else TEAL for t in coexp_hub.type]
coexp_sorted = coexp_hub.sort_values('interactions')
axes[1].barh(coexp_sorted.gene, coexp_sorted.interactions,
             color=[CORAL if t=='Ferroptosis' else TEAL for t in coexp_sorted.type],
             height=0.6, zorder=3)
for i, (_, row) in enumerate(coexp_sorted.iterrows()):
    axes[1].text(row.interactions+0.2, i, str(row.interactions),
                 va='center', fontsize=9, fontweight='bold')
axes[1].set_xlabel('Number of interactions (degree)')
axes[1].set_title('Co-expressed hub gene degrees\n(ferroptosis–apoptosis network)', fontsize=11)
axes[1].grid(axis='x', alpha=0.25)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
legend_items = [mpatches.Patch(color=CORAL, label='Ferroptosis gene'),
                mpatches.Patch(color=TEAL,  label='Apoptosis gene')]
axes[1].legend(handles=legend_items, fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nKey finding: CDKN1A has {coexp_hub[coexp_hub.gene=='CDKN1A']['interactions'].values[0]} interactions "
      f"— highest of all co-expressed hub genes")

---
## 7. Co-expression Network Analysis

A gene co-expression network was built using Pearson's correlation (PSYCH package, R).  
Genes were analysed in both positive and negative co-expression networks.

**4 ferroptosis genes were found co-expressed with apoptosis genes in both networks:**  
CDKN1A, GDF15, CISD2, NUPR1

CDKN1A and GDF15 were selected as key hub genes based on their high interaction degrees (23 and 10 respectively).

In [ ]:
# Co-expression network composition
coexp_data = pd.DataFrame({
    'network'  : ['Positive','Positive','Negative','Negative'],
    'gene_type': ['Apoptosis','Ferroptosis','Apoptosis','Ferroptosis'],
    'count'    : [223, 7, 143, 7]
})

print('Co-expression network composition:')
for net in ['Positive','Negative']:
    sub = coexp_data[coexp_data.network==net]
    total = sub['count'].sum()
    for _, row in sub.iterrows():
        pct = row['count']/total*100
        print(f"  {net} network — {row.gene_type}: {row['count']} ({pct:.1f}%)")

# Co-expression partner mapping
coexp_partners = pd.DataFrame({
    'ferroptosis_gene': ['CDKN1A']*11 + ['GDF15']*4,
    'apoptosis_gene'  : ['BTG2','CCL2','FOSL1','HSPA1B','ID1','JUN',  # positive
                         'CCL2','CKS1B','CYCS','S100A11',              # negative
                         'JUN',                                         # shared
                         'CCL2','FOSL1','IER3','IER3'],                 # GDF15
    'network'         : ['Positive','Positive','Positive','Positive','Positive','Positive',
                         'Negative','Negative','Negative','Negative','Both',
                         'Positive','Positive','Positive','Negative']
})

print(f"\nCDKN1A total apoptosis co-expression partners: "
      f"{len(coexp_partners[coexp_partners.ferroptosis_gene=='CDKN1A'])}")
print(f"GDF15 total apoptosis co-expression partners:  "
      f"{len(coexp_partners[coexp_partners.ferroptosis_gene=='GDF15'])}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, net, title in zip(axes, ['Positive','Negative'],
                          ['Positive co-expression network','Negative co-expression network']):
    data = coexp_data[coexp_data.network==net]
    total = data['count'].sum()
    bars = ax.bar(data.gene_type, data['count'],
                  color=[TEAL, CORAL], width=0.4, zorder=3)
    for bar, val in zip(bars, data['count']):
        pct = val/total*100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                f'{val}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Number of genes')
    ax.set_ylim(0, max(data['count'])*1.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Co-expression network composition\n(ferroptosis genes are a critical minority)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8. KEGG Pathway Enrichment

Functional enrichment analysis was performed using DAVID (Database for Annotation, Visualization and Integrated Discovery).  
The co-expressed hub genes were mapped to KEGG pathways to identify their carcinogenic significance.

**Most clinically significant findings:**
- CDKN1A and CYCS both appear in the **Platinum drug resistance** pathway — directly relevant to cisplatin resistance in ovarian cancer
- CDKN1A and CYCS both appear in the **p53 signalling pathway** — the central tumour suppressor axis

In [ ]:
# KEGG pathway enrichment data
kegg = pd.DataFrame({
    'pathway': ['Pathways in cancer','Colorectal cancer','Small cell lung cancer',
                'HTLV-1 infection','Renal cell carcinoma',
                'Platinum drug resistance','p53 signalling pathway'],
    'gene_count': [4,3,3,3,2,2,2],
    'genes': [
        'CDKN1A, JUN, CYCS, CKS1B',
        'CDKN1A, JUN, CYCS',
        'CDKN1A, CYCS, CKS1B',
        'FOSL1, CDKN1A, JUN',
        'CDKN1A, JUN',
        'CDKN1A, CYCS',
        'CDKN1A, CYCS'
    ],
    'clinically_significant': [False,False,False,False,False,True,True]
})

print('KEGG Pathway Enrichment Results:')
print(kegg[['pathway','gene_count','genes']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_k = [CORAL if sig else PURPLE for sig in kegg.clinically_significant]
kegg_sorted = kegg.sort_values('gene_count')
bars = ax.barh(kegg_sorted.pathway, kegg_sorted.gene_count,
               color=[CORAL if sig else PURPLE for sig in kegg_sorted.clinically_significant],
               height=0.5, zorder=3)
for bar, val, genes in zip(bars, kegg_sorted.gene_count, kegg_sorted.genes):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f'{val} ({genes})', va='center', fontsize=8.5)
ax.set_xlabel('Number of hub genes in pathway')
ax.set_xlim(0, 8)
ax.set_title('KEGG pathway enrichment of co-expressed hub genes\n(red = clinically significant to ovarian cancer therapy)',
             fontsize=12, pad=10)
legend_items = [mpatches.Patch(color=CORAL,  label='Clinically significant (OC therapy)'),
                mpatches.Patch(color=PURPLE, label='Other cancer pathway')]
ax.legend(handles=legend_items, fontsize=9)
plt.tight_layout()
plt.show()

---
## 9. Key Findings Summary

In [ ]:
print('=' * 65)
print('KEY FINDINGS SUMMARY')
print('=' * 65)
print()
print('DATASET')
print(f'  Total RNA-Seq samples collected    : 25 (15 diseased, 10 control)')
print(f'  Samples passing all QC filters     : 11 (5 diseased, 6 control)')
print(f'  Mean alignment score (final set)   : ~97% diseased, ~96% control')
print()
print('DIFFERENTIAL EXPRESSION')
print(f'  Total DEGs identified              : 7,862')
print(f'  Highly significant (p < 0.01)      : 6,577')
print(f'  Ferroptosis + Apoptosis genes      : 289')
print()
print('PPI NETWORK')
print(f'  All DEGs network                   : 236 nodes, 886 interactions')
print(f'  Ferroptosis network                : 174 nodes, 1,469 interactions')
print(f'  Top hub gene (rank 1)              : TP53')
print(f'  Key ferroptosis regulator          : GPX4 (rank 7)')
print()
print('CO-EXPRESSION NETWORK')
print(f'  Ferroptosis hub genes identified   : 4 (CDKN1A, GDF15, CISD2, NUPR1)')
print(f'  Key hub gene 1                     : CDKN1A (23 interactions)')
print(f'  Key hub gene 2                     : GDF15  (10 interactions)')
print()
print('KEGG PATHWAY ENRICHMENT')
print(f'  Pathways identified                : 7')
print(f'  Clinically significant pathways    : 2')
print(f'  Most important pathway             : Platinum drug resistance')
print(f'  Implication                        : CDKN1A may link ferroptosis')
print(f'                                       to cisplatin resistance in OC')
print()
print('=' * 65)
print('CONCLUSION')
print('=' * 65)
print()
print('CDKN1A and GDF15 identified as ferroptosis hub genes with')
print('co-regulatory roles in apoptosis. Their involvement in')
print('platinum drug resistance and p53 signalling pathways')
print('opens avenues for novel combination therapies in ovarian cancer.')
print()

---
## Citation

Banerjee, S. (2024). *Identification of ferroptosis-related hub genes and their potential relation with apoptosis in Ovarian Cancer.* MSc Dissertation, Department of Bioinformatics, Pondicherry University, Puducherry.

---

**Contact:** banerjee.shruti1306@gmail.com | [GitHub](https://github.com/shruti-banerjee)